In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor


from sklearn.decomposition import PCA

In [2]:
df=pd.read_csv('/content/gurgaon_properties_post_feature_selection_v2 (1).csv')

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


In [4]:
df['furnishing_type'].value_counts()

,count
furnishing_type,
0.0,2349
1.0,1018
2.0,187


In [5]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [6]:
df['servant room']=df['servant room'].astype(int)
df['store room']=df['store room'].astype(int)

In [7]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0,0,unfurnished,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1,0,unfurnished,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0,0,unfurnished,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1,0,semifurnished,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0,1,unfurnished,High,Mid Floor


In [8]:
X = df.drop(columns=['price'])
y = df['price']

In [9]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3554 entries, 0 to 3553
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3554 non-null   object 
 1   sector           3554 non-null   object 
 2   price            3554 non-null   float64
 3   bedRoom          3554 non-null   float64
 4   bathroom         3554 non-null   float64
 5   balcony          3554 non-null   object 
 6   agePossession    3554 non-null   object 
 7   built_up_area    3554 non-null   float64
 8   servant room     3554 non-null   int64  
 9   store room       3554 non-null   int64  
 10  furnishing_type  3554 non-null   object 
 11  luxury_category  3554 non-null   object 
 12  floor_category   3554 non-null   object 
dtypes: float64(4), int64(2), object(7)
memory usage: 361.1+ KB


# Ordinal Encoding

In [11]:
numeric_cols = ['bedRoom','bathroom','built_up_area']
binary_cols = ['servant room','store room']
categorical_cols = ['property_type','sector','balcony','agePossession','furnishing_type','luxury_category','floor_category']

In [12]:
preprocessor_ordinal = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_cols),
        ('binary','passthrough',binary_cols),
        ('ordinal', OrdinalEncoder( handle_unknown='use_encoded_value', unknown_value=-1 ), categorical_cols)],
        remainder='drop')

In [13]:
pipeline = Pipeline([
    ('preprocessor', preprocessor_ordinal),
    ('regressor', LinearRegression())
])

In [14]:
kfold = KFold( n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score( pipeline, X, y_transformed, cv=kfold, scoring='r2')
print("Mean R2:", scores.mean())
print("Std R2:", scores.std())

Mean R2: 0.7363096633436828
Std R2: 0.03238005754429933


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [16]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area']),
                                                 ('binary', 'passthrough',
                                                  ['servant room',
                                                   'store room']),
                                                 ('ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [17]:
y_pred = np.expm1(pipeline.predict(X_test))

In [18]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9463822160089357

In [19]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor_ordinal),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [20]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [21]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [22]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [23]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.893006,0.494470
5,random forest,0.881558,0.526437
6,extra trees,0.869282,0.543447
7,gradient boosting,0.872493,0.575806
9,mlp,0.814658,0.729815
4,decision tree,0.768788,0.743036
8,adaboost,0.749673,0.860296
1,svr,0.754753,0.875097
0,linear_reg,0.736310,0.946382
2,ridge,0.736314,0.946399


# OneHotEncoding

In [24]:
preprocessor_onehot = ColumnTransformer(transformers=[('num', StandardScaler(), numeric_cols),
        ('binary','passthrough',binary_cols),
        ('onehot',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),categorical_cols)],
        remainder='drop')

In [25]:
pipeline = Pipeline([
    ('preprocessor', preprocessor_onehot),
    ('regressor', LinearRegression())
])

In [26]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [27]:
scores.mean(),scores.std()

(np.float64(0.8558122985956509), np.float64(0.015558218710840512))

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [29]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area']),
                                                 ('binary', 'passthrough',
                                                  ['servant room',
                                                   'store room']),
                                                 ('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [30]:
y_pred = np.expm1(pipeline.predict(X_test))

In [31]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6483879451112142

In [32]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor_onehot),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [33]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [34]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [35]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [36]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.885346,0.478348
6,extra trees,0.886585,0.492733
1,svr,0.890468,0.524561
9,mlp,0.886865,0.528928
5,random forest,0.870692,0.538310
7,gradient boosting,0.855508,0.600026
4,decision tree,0.788744,0.633735
0,linear_reg,0.855812,0.648388
2,ridge,0.856155,0.652191
8,adaboost,0.724058,0.849662


# OneHotEncoding With PCA

In [37]:
pipeline = Pipeline([
    ('preprocessor', preprocessor_onehot),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [38]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [39]:
scores.mean(),scores.std()

(np.float64(0.7774613601631328), np.float64(0.026680442427865925))

In [40]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor_onehot),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [41]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [42]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [43]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.844295,0.593584
9,mlp,0.845481,0.613173
10,xgboost,0.835542,0.626189
5,random forest,0.829392,0.630669
1,svr,0.837736,0.642792
7,gradient boosting,0.826837,0.667025
0,linear_reg,0.777461,0.885914
2,ridge,0.777515,0.885988
8,adaboost,0.697510,0.901714
4,decision tree,0.636560,0.979523


# Target Encoder

In [44]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 2.1 MB/s eta 0:00:00


In [45]:
import category_encoders as ce

preprocessor_target = ColumnTransformer(transformers=[ ('num', StandardScaler(), numeric_cols),
                                      ('binary','passthrough',binary_cols),
                                      ('target', ce.TargetEncoder(), categorical_cols)], remainder='drop')

In [46]:
pipeline = Pipeline([
    ('preprocessor', preprocessor_target),
    ('regressor', LinearRegression())
])

In [47]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [48]:
scores.mean(),scores.std()

(np.float64(0.826403082007084), np.float64(0.018205100345016913))

In [49]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor_target),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [50]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [51]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [52]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.895690,0.469233
6,extra trees,0.894537,0.473624
10,xgboost,0.897123,0.488090
7,gradient boosting,0.885508,0.528507
4,decision tree,0.801835,0.555306
9,mlp,0.855918,0.588160
1,svr,0.866159,0.610771
8,adaboost,0.815203,0.695069
0,linear_reg,0.826403,0.716339
2,ridge,0.826401,0.716979


# Hybrid Encoding

In [53]:
preprocessor_hybrid = ColumnTransformer(transformers=[

        # Numerical
        ('num',StandardScaler(),numeric_cols),

        # Binary
        ('binary','passthrough',binary_cols),

        # Nominal categorical
        ('onehot', OneHotEncoder( drop='first', handle_unknown='ignore', sparse_output=False),[ 'property_type', 'agePossession', 'furnishing_type']),

        # Ordered categorical
        ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),['balcony','luxury_category','floor_category']),

        # High-cardinality categorical
        ('target',ce.TargetEncoder(), ['sector'])
    ],

    remainder='drop'
)

In [54]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor_hybrid),
    ('regressor', LinearRegression())
])

In [55]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [56]:
scores.mean(),scores.std()

(np.float64(0.828976628370946), np.float64(0.018936274205272736))

In [57]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [58]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area']),
                                                 ('binary', 'passthrough',
                                                  ['servant room',
                                                   'store room']),
                                                 ('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['property_type',
                                                   'agePossession',
                                                   'furnishing_type']),
                                                 ('ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['balcony', 'luxury_category',
                                                   'floor_category']),
                                                 ('target', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor', LinearRegression())])

In [59]:
y_pred = pipeline.predict(X_test)

y_pred = np.expm1(y_pred)

mae = mean_absolute_error(np.expm1(y_test), y_pred)

print("MAE:", mae)

MAE: 0.7127278833922405


In [60]:
def scorer_hybrid(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor_hybrid),
        ('regressor', model)
    ])

    # 10-Fold Cross Validation
    kfold = KFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        pipeline,
        X,
        y_transformed,
        cv=kfold,
        scoring='r2'
    )

    output.append(scores.mean())

    # Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_transformed,
        test_size=0.2,
        random_state=42
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    # Convert log price to original price
    y_pred = np.expm1(y_pred)

    mae = mean_absolute_error(
        np.expm1(y_test),
        y_pred
    )

    output.append(mae)

    return output

In [61]:
hybrid_output = []

for model_name, model in model_dict.items():

    hybrid_output.append(
        scorer_hybrid(model_name, model)
    )

In [62]:
hybrid_df = pd.DataFrame(
    hybrid_output,
    columns=['name', 'r2', 'mae']
)

hybrid_df.sort_values('mae').reset_index(drop=True)

,name,r2,mae
0,random forest,0.893504,0.468320
1,extra trees,0.892945,0.473629
2,xgboost,0.895462,0.482958
3,gradient boosting,0.883176,0.526716
4,decision tree,0.809246,0.567905
5,svr,0.864076,0.585423
6,mlp,0.854483,0.616814
7,adaboost,0.815312,0.690610
8,linear_reg,0.828977,0.712728
9,ridge,0.828992,0.713263


In [63]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 21.9 MB/s eta 0:00:00


In [64]:
import optuna

from sklearn.metrics import make_scorer

In [65]:
def original_scale_mae(y_true_log, y_pred_log):

    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)

    return mean_absolute_error(y_true, y_pred)


mae_scorer = make_scorer(
    original_scale_mae,
    greater_is_better=False
)

In [66]:
def objective(trial):

    rf = RandomForestRegressor(

        n_estimators=trial.suggest_int(
            'n_estimators', 200, 1200
        ),

        max_depth=trial.suggest_int(
            'max_depth', 5, 30
        ),

        min_samples_split=trial.suggest_int(
            'min_samples_split', 2, 15
        ),

        min_samples_leaf=trial.suggest_int(
            'min_samples_leaf', 1, 8
        ),

        max_features=trial.suggest_float(
            'max_features', 0.5, 1.0
        ),

        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor_hybrid),
        ('regressor', rf)
    ])

    kfold = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        pipeline,
        X,
        y_transformed,
        cv=kfold,
        scoring=mae_scorer,
        n_jobs=-1
    )

    return -scores.mean()

In [68]:
study_rf = optuna.create_study(
    direction='minimize'
)

study_rf.optimize(
    objective,
    n_trials=50
)

[I 2026-08-13 05:03:59,168] A new study created in memory with name: no-name-26d7929a-0c4c-4385-aab6-de836b0725de
[I 2026-08-13 05:04:11,295] Trial 0 finished with value: 0.5078514964418641 and parameters: {'n_estimators': 281, 'max_depth': 29, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 0.5383665697551006}. Best is trial 0 with value: 0.5078514964418641.
[I 2026-08-13 05:04:56,737] Trial 1 finished with value: 0.5154360104079069 and parameters: {'n_estimators': 916, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 0.8341983603392836}. Best is trial 0 with value: 0.5078514964418641.
[I 2026-08-13 05:05:08,248] Trial 2 finished with value: 0.5219293070509892 and parameters: {'n_estimators': 388, 'max_depth': 11, 'min_samples_split': 8, 'min_samples_leaf': 4, 'max_features': 0.7939029506090965}. Best is trial 0 with value: 0.5078514964418641.
[I 2026-08-13 05:05:20,651] Trial 3 finished with value: 0.5043617261052754 and parameters: {'n_e

In [74]:
print("Best CV MAE:", study_rf.best_value)

study_rf.best_params

Best CV MAE: 0.49378521310165774


{'n_estimators': 460,
 'max_depth': 29,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 0.7613725507086561}

In [75]:
best_rf = RandomForestRegressor(
    **study_rf.best_params,
    random_state=42,
    n_jobs=-1
)

best_rf_pipeline = Pipeline([
    ('preprocessor', preprocessor_hybrid),
    ('regressor', best_rf)
])

In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

In [77]:
best_rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area']),
                                                 ('binary', 'passthrough',
                                                  ['servant room',
                                                   'store room']),
                                                 ('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['property_type',
                                                   'agePossession',
                                                   'furnishing_type']),
                                                 ('ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['balcony', 'luxury_category',
                                                   'floor_category']),
                                                 ('target', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=29,
                                       max_features=0.7613725507086561,
                                       n_estimators=460, n_jobs=-1,
                                       random_state=42))])

In [78]:
y_pred = best_rf_pipeline.predict(X_test)

y_pred_original = np.expm1(y_pred)
y_test_original = np.expm1(y_test)

In [79]:
rf_mae = mean_absolute_error(
    y_test_original,
    y_pred_original
)

print("Tuned Random Forest MAE:", rf_mae)

Tuned Random Forest MAE: 0.46344225033261105


In [80]:
def objective_xgb(trial):

    xgb = XGBRegressor(

        n_estimators=trial.suggest_int(
            'n_estimators', 300, 1500
        ),

        learning_rate=trial.suggest_float(
            'learning_rate',
            0.01, 0.15,
            log=True
        ),

        max_depth=trial.suggest_int(
            'max_depth', 3, 10
        ),

        min_child_weight=trial.suggest_int(
            'min_child_weight', 1, 10
        ),

        subsample=trial.suggest_float(
            'subsample', 0.6, 1.0
        ),

        colsample_bytree=trial.suggest_float(
            'colsample_bytree', 0.6, 1.0
        ),

        reg_alpha=trial.suggest_float(
            'reg_alpha',
            1e-4, 10,
            log=True
        ),

        reg_lambda=trial.suggest_float(
            'reg_lambda',
            1e-3, 20,
            log=True
        ),

        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor_hybrid),
        ('regressor', xgb)
    ])

    kfold = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_val_score(
        pipeline,
        X,
        y_transformed,
        cv=kfold,
        scoring=mae_scorer,
        n_jobs=-1
    )

    return -scores.mean()

In [81]:
study_xgb = optuna.create_study(
    direction='minimize'
)

study_xgb.optimize(
    objective_xgb,
    n_trials=50
)

[I 2026-08-13 05:19:56,157] A new study created in memory with name: no-name-13fe2e5b-12a2-4491-bce3-4a532c4459f9
[I 2026-08-13 05:20:01,596] Trial 0 finished with value: 0.4893713001930708 and parameters: {'n_estimators': 1233, 'learning_rate': 0.0201567370733828, 'max_depth': 6, 'min_child_weight': 9, 'subsample': 0.9110464756220114, 'colsample_bytree': 0.8436395010296067, 'reg_alpha': 0.015374988535416542, 'reg_lambda': 6.4464026836082144}. Best is trial 0 with value: 0.4893713001930708.
[I 2026-08-13 05:20:03,461] Trial 1 finished with value: 0.5241978676524711 and parameters: {'n_estimators': 1125, 'learning_rate': 0.06124999882124916, 'max_depth': 3, 'min_child_weight': 6, 'subsample': 0.7154670945676832, 'colsample_bytree': 0.6051504927714474, 'reg_alpha': 2.589983336212708, 'reg_lambda': 0.36039335435715586}. Best is trial 0 with value: 0.4893713001930708.
[I 2026-08-13 05:20:09,049] Trial 2 finished with value: 0.49824553761817947 and parameters: {'n_estimators': 1196, 'learni

In [82]:
print("Best XGBoost CV MAE:", study_xgb.best_value)

study_xgb.best_params

Best XGBoost CV MAE: 0.47331591538898826


{'n_estimators': 1146,
 'learning_rate': 0.02338113889216173,
 'max_depth': 7,
 'min_child_weight': 1,
 'subsample': 0.97285368437607,
 'colsample_bytree': 0.8249995774808797,
 'reg_alpha': 0.011003164357885927,
 'reg_lambda': 0.05373107321813057}

In [83]:
best_xgb = XGBRegressor(
    **study_xgb.best_params,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

best_xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_hybrid),
    ('regressor', best_xgb)
])

In [84]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

best_xgb_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area']),
                                                 ('binary', 'passthrough',
                                                  ['servant room',
                                                   'store room']),
                                                 ('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['property_type',
                                                   'agePossession',
                                                   'furnishing_type']),
                                                 ('ordinal',
                                                  OrdinalEncoder(handle_u...
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None,
                              learning_rate=0.02338113889216173, max_bin=None,
                              max_cat_threshold=None, max_cat_to_onehot=None,
                              max_delta_step=None, max_depth=7, max_leaves=None,
                              min_child_weight=1, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=1146, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [85]:
y_pred_xgb = best_xgb_pipeline.predict(X_test)

xgb_mae = mean_absolute_error(
    np.expm1(y_test),
    np.expm1(y_pred_xgb)
)

print("Tuned XGBoost MAE:", xgb_mae)

Tuned XGBoost MAE: 0.4428102832873327


In [86]:
# Convert back to original price scale
actual_price = np.expm1(y_test)
predicted_price = np.expm1(y_pred_xgb)

error_df = X_test.copy()

error_df['actual_price'] = actual_price.values
error_df['predicted_price'] = predicted_price

error_df['absolute_error'] = abs(
    error_df['actual_price'] -
    error_df['predicted_price']
)

error_df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,actual_price,predicted_price,absolute_error
299,flat,sector 90,4.0,4.0,3+,Moderately Old,2005.0,1,0,unfurnished,Low,High Floor,1.70,1.703106,0.003106
2240,flat,sector 92,2.0,2.0,2,New Property,1083.0,0,0,semifurnished,Low,Low Floor,0.78,0.758386,0.021614
2384,flat,sector 37d,3.0,3.0,2,Under Construction,1439.0,0,0,unfurnished,Low,Low Floor,1.10,1.238161,0.138161
3473,house,sector 23,4.0,4.0,2,Old Property,3078.0,0,0,unfurnished,Low,Low Floor,5.00,5.481075,0.481075
2614,flat,sector 102,4.0,5.0,3+,Moderately Old,2311.0,1,0,unfurnished,Medium,High Floor,2.65,2.553311,0.096689


In [87]:
error_df.groupby('property_type')['absolute_error'].agg(
    ['count', 'mean', 'median']
)

,count,mean,median
property_type,,,
flat,573,0.270308,0.133691
house,138,1.159069,0.588352


In [88]:
error_df.sort_values(
    'absolute_error',
    ascending=False
).head(20)

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,actual_price,predicted_price,absolute_error
1271,house,sector 43,7.0,8.0,3+,Old Property,9000.0,1,1,semifurnished,Low,Mid Floor,11.50,22.819317,11.319317
3066,house,sector 60,9.0,12.0,3+,Relatively New,6390.0,1,0,semifurnished,Medium,Mid Floor,18.02,7.088709,10.931291
2860,house,sector 25,6.0,5.0,2,Relatively New,6000.0,0,0,unfurnished,Low,Mid Floor,16.00,6.204030,9.795970
76,flat,sector 48,4.0,4.0,3+,Moderately Old,3831.0,1,1,semifurnished,Medium,Mid Floor,9.30,4.465716,4.834284
985,house,sector 43,9.0,9.0,3+,Moderately Old,3510.0,1,0,unfurnished,Low,Mid Floor,14.00,9.378872,4.621128
1655,house,sector 50,5.0,7.0,2,Moderately Old,7000.0,1,0,furnished,Low,Mid Floor,10.00,14.109781,4.109781
3044,house,sector 26,5.0,4.0,3+,Moderately Old,4518.0,1,1,semifurnished,Medium,Mid Floor,18.00,14.088240,3.911760
1295,flat,sector 54,3.0,3.0,3,Moderately Old,1382.0,0,0,unfurnished,Low,Low Floor,5.70,1.979159,3.720841
3307,flat,sector 48,5.0,6.0,3+,Relatively New,7444.0,1,0,furnished,Low,High Floor,15.00,11.358463,3.641537
443,flat,sector 106,5.0,6.0,3+,Under Construction,3797.0,0,0,semifurnished,Medium,Mid Floor,6.30,2.742476,3.557524


In [89]:
error_df.groupby('property_type')['absolute_error'].agg(
    count='count',
    mean_mae='mean',
    median_error='median',
    max_error='max'
)

,count,mean_mae,median_error,max_error
property_type,,,,
flat,573,0.270308,0.133691,4.834284
house,138,1.159069,0.588352,11.319317


In [90]:
error_df['price_range'] = pd.cut(
    error_df['actual_price'],
    bins=[0, 1, 2, 3, 5, 10, np.inf],
    labels=[
        '<1 Cr',
        '1-2 Cr',
        '2-3 Cr',
        '3-5 Cr',
        '5-10 Cr',
        '10+ Cr'
    ]
)

error_df.groupby(
    'price_range',
    observed=True
)['absolute_error'].agg(
    ['count', 'mean', 'median']
)

,count,mean,median
price_range,,,
<1 Cr,202,0.145560,0.092163
1-2 Cr,264,0.208714,0.130619
2-3 Cr,97,0.375307,0.256721
3-5 Cr,61,0.771038,0.591553
5-10 Cr,65,1.209415,0.921398
10+ Cr,22,3.103836,2.261167
